# 02c - Build User Profiles (Mean)
Create and save user vectors + user histories from train interactions using imported MeanProfileManager.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import drive
import sys
import os
%pip install -q faiss-cpu
%pip install -q duckdb polars

sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
sys.path.insert(0,'/content/drive/My Drive/Thesis/book_recsys/app')

from app.recommenders.vector_recommender import VectorRecommender
from app.recommenders.processer import BookProcessor, UserProcessor

In [3]:
!pip install -q polars numpy tqdm cupy-cuda12x faiss-cpu

In [4]:
_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

In [5]:
import duckdb

class BookProcessor:
    def __init__(self, item_index_path: str):
        print(f"Loading Book mapping from {item_index_path} ...")

        query = f"SELECT CAST(book_id AS VARCHAR), CAST(COALESCE(row_idx, book_id) AS INTEGER) FROM read_csv_auto('{item_index_path}')"

        with duckdb.connect(':memory:') as conn:
            results = conn.execute(query).fetchall()

        self.id_to_idx = {str(row[0]): int(row[1]) for row in results}
        self.idx_to_id = {int(row[1]): str(row[0]) for row in results}

        self.total_books = len(self.id_to_idx)
        print(f" -> Loaded {self.total_books:,} books into RAM dictionary.")

    def get_idx(self, book_id: str) -> int:
        return self.id_to_idx.get(str(book_id), -1)

    def get_id(self, item_idx: int) -> str:
        return self.idx_to_id.get(item_idx, None)
book_processor = BookProcessor(item_index_path= imtem_index)
user_processor = UserProcessor(user_index_path= user_index,history_db_path= history_db)


cb_recommend = VectorRecommender(
    book_processor= book_processor,
    user_processor= user_processor,
    faiss_index_path=sbert_item_faiss_index,
    user_vectors_path = sbert_user_matrix,
    # item_faiss_index_path=sbert_item_faiss_index
)





Loading Book mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_book_index.csv ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> Loaded 1,173,713 books into RAM dictionary.
Loading User mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv...
Connecting to Persistent User History DB...
Loading Data & CPU FAISS Index...
Recommender Engine Ready (Users: 743,401)


In [8]:
cf_recommend = VectorRecommender(
    book_processor= book_processor,
    user_processor= user_processor,
    faiss_index_path=ials_item_faiss_index,
    user_vectors_path = ials_user_matrix,
    # item_faiss_index_path=sbert_item
)

Loading Data & CPU FAISS Index...
Recommender Engine Ready (Users: 743,401)


In [15]:
print("CB "+ "="*5)
print(cb_recommend.recommend_realtime("000258cdb856ee85203323147d1223dd",100))

print("CF "+ "="*5)
print(cf_recommend.recommend_realtime("000258cdb856ee85203323147d1223dd",100))

CB =====
[RecommendItem(item_id='12987033', score=0.7861018180847168, rank=1), RecommendItem(item_id='17450037', score=0.7859830856323242, rank=2), RecommendItem(item_id='1332498', score=0.7846333384513855, rank=3), RecommendItem(item_id='70950', score=0.7805496454238892, rank=4), RecommendItem(item_id='19273827', score=0.7800249457359314, rank=5), RecommendItem(item_id='15837529', score=0.7782104015350342, rank=6), RecommendItem(item_id='13809', score=0.775473415851593, rank=7), RecommendItem(item_id='13493898', score=0.7717981338500977, rank=8), RecommendItem(item_id='694119', score=0.7699168920516968, rank=9), RecommendItem(item_id='27782996', score=0.7693873047828674, rank=10), RecommendItem(item_id='25673410', score=0.7690628170967102, rank=11), RecommendItem(item_id='17258996', score=0.7662659883499146, rank=12), RecommendItem(item_id='24991544', score=0.765531063079834, rank=13), RecommendItem(item_id='24292310', score=0.765531063079834, rank=14), RecommendItem(item_id='26218032